# Classify Documents With the API

Use this notebook after starting the FastAPI server locally:

```powershell
python -m uvicorn app.main:app --host 127.0.0.1 --port 8000
```

Then edit `my_text` below and run the cells.

In [2]:
import json
from urllib import request, error

API_URL = "http://127.0.0.1:8000"
CLASSIFY_ENDPOINT = f"{API_URL}/classify_document"
HEALTH_ENDPOINT = f"{API_URL}/health"

## 1. Check That the API Is Running

In [3]:
def get_json(url):
    with request.urlopen(url, timeout=10) as response:
        return json.loads(response.read().decode("utf-8"))

try:
    health = get_json(HEALTH_ENDPOINT)
    print("API is running:")
    print(json.dumps(health, indent=2))
except Exception as exc:
    print("API is not reachable. Start it with:")
    print("python -m uvicorn app.main:app --host 127.0.0.1 --port 8000")
    raise exc

API is running:
{
  "status": "ok",
  "app_name": "Trellis Document Classifier",
  "version": "1.0.0"
}


## 2. Write Your Text

In [4]:
my_text = """
The team won the championship after a dramatic final match.
The coach praised the players for their discipline throughout the season.
""".strip()

my_text

'The team won the championship after a dramatic final match.\nThe coach praised the players for their discipline throughout the season.'

## 3. Classify the Text

In [5]:
def classify_document(document_text):
    payload = json.dumps({"document_text": document_text}).encode("utf-8")
    api_request = request.Request(
        CLASSIFY_ENDPOINT,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with request.urlopen(api_request, timeout=30) as response:
            return json.loads(response.read().decode("utf-8"))
    except error.HTTPError as exc:
        error_body = exc.read().decode("utf-8")
        print(f"API returned HTTP {exc.code}")
        print(error_body)
        raise


result = classify_document(my_text)
print(json.dumps(result, indent=2))

{
  "message": "Classification successful",
  "label": "sport",
  "confidence": 0.7913,
  "raw_label": "sport",
  "is_other": false
}


## 4. Classify Multiple Texts

In [28]:
texts = [
    "NASA announced a new telescope mission to study distant galaxies.",
    "Doctors tested a new treatment in a clinical hospital study.",
    "The company reported strong revenue growth after new investments.",
    "The team won the championship after a dramatic final match."
]

for text in texts:
    prediction = classify_document(text)
    print("Text:", text)
    print("Prediction:", prediction["label"], "| confidence:", prediction["confidence"])
    print("-" * 80)

Text: NASA announced a new telescope mission to study distant galaxies.
Prediction: space | confidence: 0.8609
--------------------------------------------------------------------------------
Text: Doctors tested a new treatment in a clinical hospital study.
Prediction: medical | confidence: 0.845
--------------------------------------------------------------------------------
Text: The company reported strong revenue growth after new investments.
Prediction: business | confidence: 0.8794
--------------------------------------------------------------------------------
Text: The team won the championship after a dramatic final match.
Prediction: sport | confidence: 0.6864
--------------------------------------------------------------------------------


In [29]:
classify_document("The team won the championship after a dramatic final match.")

{'message': 'Classification successful',
 'label': 'sport',
 'confidence': 0.6864,
 'raw_label': 'sport',
 'is_other': False}

# 5. Using provided data

In [7]:
import os
import re
import pandas as pd

In [ ]:
path_to_data = '../data/full_data/'

data = list()

for folder in os.listdir(path_to_data):
    path_to_folder = os.path.join(path_to_data, folder)
    if os.path.isdir(path_to_folder):
        for file in os.listdir(path_to_folder):
            if file.endswith('.txt'):
                path_to_file = os.path.join(path_to_folder, file)
                with open(path_to_file, 'r', encoding='utf-8') as f:
                    text = f.read()
                    data.append({
                        'folder': folder,
                        'file': file,
                        'text': text
                    })

df_data = pd.DataFrame(data)
df_data.head()

,folder,file,text
0,business,business_1.txt,Lufthansa flies back to profit\n\nGerman airli...
1,business,business_10.txt,Winn-Dixie files for bankruptcy\n\nUS supermar...
2,business,business_100.txt,US economy still growing says Fed\n\nMost area...
3,business,business_11.txt,Saab to build Cadillacs in Sweden\n\nGeneral M...
4,business,business_12.txt,Bank voted 8-1 for no rate change\n\nThe decis...


In [25]:
results = []
for index, row in df_data.iterrows():
    text = row['text']
    prediction = classify_document(text)
    results.append({
        'folder': row['folder'],
        'file': row['file'],
        'text': text,
        'label': prediction['label'],
        'confidence': prediction['confidence'],
        'prediction_details': prediction
    })
results[0]

{'folder': 'business',
 'file': 'business_1.txt',
 'text': 'Lufthansa flies back to profit\n\nGerman airline Lufthansa has returned to profit in 2004 after posting huge losses in 2003.\n\nIn a preliminary report, the airline announced net profits of 400m euros ($527.61m; £274.73m), compared with a loss of 984m euros in 2003. Operating profits were at 380m euros, ten times more than in 2003. Lufthansa was hit in 2003 by tough competition and a dip in demand following the Iraq war and the killer SARS virus. It was also hit by troubles at its US catering business. Last year, Lufthansa showed signs of recovery even as some European and US airlines were teetering on the brink of bankruptcy. The board of Lufthansa has recommended paying a 2004 dividend of 0.30 euros per share. In 2003, shareholders did not get a dividend. The company said that it will give all the details of its 2004 results on 23 March.\n',
 'label': 'business',
 'confidence': 0.9088,
 'prediction_details': {'message': 'Cla

In [26]:
df_results = pd.DataFrame(results)
df_results

,folder,file,text,label,confidence,prediction_details
0,business,business_1.txt,Lufthansa flies back to profit\n\nGerman airli...,business,0.9088,"{'message': 'Classification successful', 'labe..."
1,business,business_10.txt,Winn-Dixie files for bankruptcy\n\nUS supermar...,business,0.8102,"{'message': 'Classification successful', 'labe..."
2,business,business_100.txt,US economy still growing says Fed\n\nMost area...,business,0.9492,"{'message': 'Classification successful', 'labe..."
3,business,business_11.txt,Saab to build Cadillacs in Sweden\n\nGeneral M...,business,0.9531,"{'message': 'Classification successful', 'labe..."
4,business,business_12.txt,Bank voted 8-1 for no rate change\n\nThe decis...,business,0.9472,"{'message': 'Classification successful', 'labe..."
...,...,...,...,...,...,...
1001,technologie,technologie_95.txt,Mobile games come of age\n\nThe BBC News websi...,technologie,0.9320,"{'message': 'Classification successful', 'labe..."
1002,technologie,technologie_96.txt,California sets fines for spyware\n\nThe maker...,technologie,0.9508,"{'message': 'Classification successful', 'labe..."
1003,technologie,technologie_97.txt,Web helps collect aid donations\n\nThe web is ...,technologie,0.9381,"{'message': 'Classification successful', 'labe..."
1004,technologie,technologie_98.txt,Mobiles rack up 20 years of use\n\nMobile phon...,technologie,0.9631,"{'message': 'Classification successful', 'labe..."


In [27]:
from sklearn.metrics import classification_report

y_true = df_results['folder']
y_pred = df_results['label']
print(classification_report(y_true, y_pred))

               precision    recall  f1-score   support

     business       0.99      1.00      1.00       100
entertainment       1.00      1.00      1.00       100
         food       1.00      1.00      1.00       100
     graphics       1.00      0.98      0.99       100
   historical       1.00      1.00      1.00       100
      medical       1.00      1.00      1.00       100
        other       0.50      0.67      0.57         6
     politics       1.00      0.99      0.99       100
        space       0.99      0.99      0.99       100
        sport       0.99      1.00      1.00       100
  technologie       0.99      0.98      0.98       100

     accuracy                           0.99      1006
    macro avg       0.95      0.96      0.96      1006
 weighted avg       0.99      0.99      0.99      1006

